# 连接华为 MRS Kerberos 安全集群的 Hive（pyhive —— 不需要改源码）

**结论：不用改 pyhive 的源码。** 用 pyhive 官方逃生口 `hive.connect(thrift_transport=...)` 把
「TCP 连接的地址」和「Kerberos SPN 的 host 部分」解耦即可（见第 4 格）。

## 原理：原代码为什么连不上

pyhive 0.7.0 在 `auth="KERBEROS"` 时，把 **TCP 连接用的 host（IP）直接当成 SPN 的 host 部分**
（源码 `get_installed_sasl(host=host, ...)`），即向 KDC 请求 `hive/10.0.0.51@REALM` 的票据；
而华为 MRS 注册的 Hive 服务票据是**固定串**：

```
hive/hadoop.haddop_252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com@252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
```

中间段**不是** `_HOST`、**不是** IP/主机名，规则是 `hadoop. + 域名全小写`（华为与开源实现的差异点）。
另外，`hive.Connection` 根本没有 `krbhost` 参数（原代码会直接 `TypeError`），也没有任何参数能单独指定 SPN。

## 正确姿势（本 notebook 全部替你做好）

1. **自定义 transport**：TCP 层连 `10.0.0.51:21066`；SASL 层的 host 传 SPN 中间段 `hadoop.haddop_xxx.com`；
2. **krb5 配置**：生成 `krb5.ini` / `krb5.conf` 并设 `KRB5_CONFIG`，其中 `dns_canonicalize_hostname = false`
   是关键 —— SPN 中间段是个 DNS 里不存在的"假域名"，必须禁止 Kerberos 客户端去解析它；
3. **用户票据（TGT）**：三选一 —— keytab kinit / 交互式 kinit / Windows 密码直连（winkerberos SSPI 显式账密，免 kinit）。

## ⚠️ 需要你确认的一处

按"hadoop. + 域名全小写"推导，中间段应为 `hadoop.252a63ec_...com`；但你给出的完整 ST 带 `haddop_` 前缀。
第 4 格会把两个候选**依次自动尝试**，连上后请到 MRS Manager → Hive → 配置 → 全部配置 →
搜 `hive.server2.authentication.kerberos.principal` 核对权威值，然后删掉错误的候选。

## 运行环境

| 环境 | 说明 |
|---|---|
| Linux（同 VPC 的 ECS / ModelArts） | `kinit` + keytab，最稳 |
| Windows 本机 | 装 MIT Kerberos for Windows（kinit 用）；或第 1 格填 `KERBEROS_PASSWORD` 走 SSPI 密码直连（前提：Windows 能定位 KDC —— VPC DNS 有 SRV 记录，或管理员执行过 `ksetup /addkdc <REALM> <Master IP>`） |

> 当前这台机器连 `10.0.0.51:21066` 超时 —— 先解决网络（VPN / 安全组 / 换到集群同 VPC 环境再跑）。


In [1]:
# ================== 1. 连接配置（按实际情况修改） ==================
HIVE_HOST = "110.239.86.94" # HiveServer2 节点 EIP（TCP 层连这里）
HIVE_PORT = 21066           # HiveServer2 Thrift 端口
DATABASE  = "default"
USERNAME  = "hhx"           # MRS 业务用户

# ---- Kerberos：华为 MRS 专有规则 ----
REALM   = "252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM"   # 系统域名（Realm）
SERVICE = "hive"

# 华为 MRS 的 Hive SPN 中间段不是 _HOST、不是 IP，而是固定串：hadoop. + 域名全小写
# 你给出的完整 ST：hive/hadoop.haddop_252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com@252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
# 注意：它带 haddop_ 前缀，与“hadoop.+域名全小写”推导出的 hadoop.252a63ec_... 略有出入 ——
#       两个候选会依次自动尝试；确认后在 MRS Manager(Hive->配置->搜 principal) 核对并删掉错的
SPN_HOST_CANDIDATES = [
    "hadoop.haddop_" + REALM.lower(),   # ① 你给出的完整 ST 的中间段（首选）
    "hadoop." + REALM.lower(),          # ② 按规则 hadoop.+域名全小写 推导
]

# ---- KDC：Master 节点 IP（MRS 控制台 -> 集群 -> 节点管理 -> Master，建议都写上）----
KDC_HOSTS = ["110.239.86.94"]   # Master 节点 EIP

# ---- 用户凭据（三选一，见第 3 格）----
USER_PRINCIPAL    = f"{USERNAME}@{REALM}"        # 人机用户 principal
USER_KEYTAB       = r"D:\path\to\hhx.keytab"  # (A) MRS Manager 下载的 keytab 路径（Linux 上改成 Linux 路径）
KERBEROS_PASSWORD = None                         # (B) Windows 密码直连模式：填密码字符串即启用

for i, h in enumerate(SPN_HOST_CANDIDATES, 1):
    print(f"候选 SPN {i}: {SERVICE}/{h}@{REALM}")

候选 SPN 1: hive/hadoop.haddop_252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com@252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
候选 SPN 2: hive/hadoop.252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com@252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM


In [2]:
# ================== 2. 依赖安装 + 兼容补丁 + 连通性检查 ==================
import importlib, socket, subprocess, sys

def ensure_module(name):
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"[pip install] {name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])

for m in ("thrift_sasl", "puresasl"):   # 注意: pip 包叫 pure-sasl，但导入名是 puresasl
    ensure_module(m)

if sys.platform == "win32":
    # Windows: 用 winkerberos(SSPI) 给 pure-sasl 提供 GSSAPI
    ensure_module("winkerberos")
    # 兼容补丁 —— 必须在 pure-sasl 被导入【之前】执行，否则直接 AttributeError:
    # pure-sasl 0.6.2 找 winkerberos.authGSSClientUsername(小写 n)，
    # 而 winkerberos>=0.8 已改名为 authGSSClientUserName —— 运行时补个别名即可，无需改库文件
    import winkerberos
    if not hasattr(winkerberos, "authGSSClientUsername"):
        winkerberos.authGSSClientUsername = winkerberos.authGSSClientUserName
    print("winkerberos 兼容补丁 OK")
else:
    # Linux: pure-sasl 的 GSSAPI 需要 pykerberos 或 cyrus-sasl 之一
    for m in ("kerberos", "sasl"):
        try:
            importlib.import_module(m)
            print(f"使用原生 Kerberos 绑定: {m}")
            break
        except ImportError:
            continue
    else:
        print("! 需要原生绑定：pip install kerberos（需 gcc + libkrb5-dev）或 conda install -c conda-forge sasl")

# 连通性：HiveServer2 + KDC 都要通
targets = [(HIVE_HOST, HIVE_PORT, "HiveServer2")] + [(h, 88, f"KDC {h}") for h in KDC_HOSTS]
for host, port, name in targets:
    s = socket.socket(); s.settimeout(5)
    try:
        s.connect((host, port)); print(f"[OK]   {name} {host}:{port} 可达")
    except Exception as e:
        print(f"[FAIL] {name} {host}:{port} 不可达: {e}")
    finally:
        s.close()

winkerberos 兼容补丁 OK
[OK]   HiveServer2 110.239.86.94:21066 可达


[FAIL] KDC 110.239.86.94 110.239.86.94:88 不可达: [WinError 10061] 由于目标计算机积极拒绝，无法连接。


In [3]:
# ================== 3. 生成 krb5 配置 + 获取用户票据(TGT) ==================
# dns_canonicalize_hostname = false 是关键：SPN 中间段 hadoop.haddop_xxx.com 是个
# DNS 里不存在的“假域名”，必须禁止 Kerberos 客户端对它做 DNS 解析，原样拿去当 SPN。
from pathlib import Path
import os, shutil, subprocess

WORKDIR   = Path.cwd()
KRB5_FILE = WORKDIR / ("krb5.ini" if sys.platform == "win32" else "krb5.conf")

lines = [
    "[libdefaults]",
    f"    default_realm = {REALM}",
    "    dns_canonicalize_hostname = false",   # <- 关键！
    "    rdns = false",
    "    udp_preference_limit = 1",            # 票据大时走 TCP
    "",
    "[realms]",
    f"    {REALM} = {{",
    *[f"        kdc = {h}" for h in KDC_HOSTS],
    f"        admin_server = {KDC_HOSTS[0]}",
    "    }",
    "",
    "[domain_realm]",
    f"    .{REALM.lower()} = {REALM}",
    f"    .haddop_{REALM.lower()} = {REALM}",
    f"    hadoop.haddop_{REALM.lower()} = {REALM}",
    "",
]
KRB5_FILE.write_text("\n".join(lines), encoding="utf-8", newline="\n")
os.environ["KRB5_CONFIG"] = str(KRB5_FILE)   # 之后 kinit / cyrus-sasl / pykerberos 都会读它
                                             # （注意：Windows SSPI 不读它，SSPI 用 Windows 自己的 KDC 定位）
print(f"已生成: {KRB5_FILE}\n" + "\n".join(lines))

def find_kerberos_tool(name):
    p = shutil.which(name)
    if p and "system32" not in p.lower():     # 排除 Windows 自带的 LSASS klist（不是 KfW 的）
        return p
    for d in (r"C:\Program Files\MIT Kerberos\bin", r"C:\Program Files (x86)\MIT Kerberos\bin"):
        c = Path(d) / f"{name}.exe"
        if c.exists():
            return str(c)
    return None

env    = {**os.environ, "KRB5_CONFIG": str(KRB5_FILE)}
keytab = USER_KEYTAB if USER_KEYTAB and Path(USER_KEYTAB).exists() else None

if sys.platform == "win32" and KERBEROS_PASSWORD:
    print("\n[密码模式] 将用 winkerberos/SSPI 账号+密码直接认证，无需 kinit。")
    print(f"前提：Windows 能定位 KDC —— VPC DNS 有 _kerberos._tcp.{REALM.lower()} SRV 记录，")
    print(f"      或管理员执行过:  ksetup /setrealm {REALM}  然后  ksetup /addkdc {REALM} <Master节点IP>")
elif (kinit := find_kerberos_tool("kinit")) is None:
    print("\n未找到 kinit ——")
    if sys.platform == "win32":
        print("  方案A: 安装 MIT Kerberos for Windows (https://web.mit.edu/kerberos/) 后重跑本格")
        print("  方案B: 第 1 格填 KERBEROS_PASSWORD 走密码直连模式")
    else:
        print("  Linux 请先执行:  yum install krb5-workstation  或  apt install krb5-user")
else:
    klist = find_kerberos_tool("klist")
    r = subprocess.run([klist], capture_output=True, text=True, env=env) if klist else None
    if r and r.returncode == 0 and "krbtgt" in r.stdout:
        print("\n已有有效票据，跳过 kinit：")
        print(r.stdout)
    elif keytab:
        subprocess.run([kinit, "-kt", str(keytab), USER_PRINCIPAL], check=True, env=env)
        print(f"\nkinit(keytab) 完成: {USER_PRINCIPAL}")
    else:
        print(f"\n请在终端执行一次（输入 MRS 密码）后重跑本格:  kinit {USER_PRINCIPAL}")
        print("（或在第 1 格配置 USER_KEYTAB 指向从 MRS Manager 下载的 keytab 后重跑）")

已生成: D:\soft\xgboos_demo\0817_new_dev\xgboost-modelarts-demo\hive_export\krb5.ini
[libdefaults]
    default_realm = 252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
    dns_canonicalize_hostname = false
    rdns = false
    udp_preference_limit = 1

[realms]
    252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM = {
        kdc = 110.239.86.94
        admin_server = 110.239.86.94
    }

[domain_realm]
    .252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com = 252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
    .haddop_252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com = 252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
    hadoop.haddop_252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com = 252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM


未找到 kinit ——
  方案A: 安装 MIT Kerberos for Windows (https://web.mit.edu/kerberos/) 后重跑本格
  方案B: 第 1 格填 KERBEROS_PASSWORD 走密码直连模式


In [4]:
# ================== 4. 连接（核心：不改 pyhive 源码的写法） ==================
# pyhive 的 KERBEROS 分支写死了 sasl.host = TCP host → SPN = hive@10.0.0.51，
# 且 connect() 没有任何参数能单独指定 SPN（krbhost 参数不存在）。
# 官方逃生口: thrift_transport= 参数 —— TCP 层连真实 IP，SASL 层 host 传华为固定 SPN 中间段。
from urllib.parse import quote
from thrift.transport import TSocket
import thrift_sasl
from pyhive import hive
from pyhive.hive import get_installed_sasl
from pyhive.sasl_compat import PureSASLClient

def make_transport(spn_host):
    tcp_sock = TSocket.TSocket(HIVE_HOST, HIVE_PORT)
    tcp_sock.setTimeout(30000)

    def sasl_factory():
        if sys.platform == "win32" and KERBEROS_PASSWORD:
            # Windows 密码直连：SSPI 显式凭据（winkerberos 支持 "user@REALM:password"）
            pw_principal = f"{USER_PRINCIPAL}:{quote(KERBEROS_PASSWORD)}"
            return PureSASLClient(host=spn_host, service=SERVICE,
                                  mechanism="GSSAPI", principal=pw_principal)
        # Linux / 已 kinit：优先 pyhive 自带工厂（cyrus-sasl；缺了自动退回 pure-sasl）
        return get_installed_sasl(host=spn_host, sasl_auth="GSSAPI", service=SERVICE)

    return thrift_sasl.TSaslClientTransport(sasl_factory, "GSSAPI", tcp_sock)

conn = None
for spn_host in SPN_HOST_CANDIDATES:
    principal = f"{SERVICE}/{spn_host}@{REALM}"
    print(f"尝试 SPN: {principal}")
    t = make_transport(spn_host)
    try:
        conn = hive.connect(thrift_transport=t, database=DATABASE, username=USERNAME)
        print(">>> 连接成功！生效 SPN =", principal)
        SPN_HOST_IN_USE = spn_host
        break
    except Exception as e:
        print(f">>> 失败: {type(e).__name__}: {e}\n")
        try:
            t.close()
        except Exception:
            pass

if conn is None:
    raise RuntimeError(
        "所有 SPN 候选均失败，排查顺序：\n"
        " 1) 票据: klist 确认有 krbtgt（密码模式确认密码正确）\n"
        " 2) SPN : MRS Manager -> Hive -> 配置 -> 搜 principal，把准确值填进 SPN_HOST_CANDIDATES\n"
        " 3) 网络: 本机到 HIVE_HOST:21066 与 KDC:88 是否可达\n"
        " 4) 端口: MRS 上 HiveServer2 thrift 端口是否为 21066"
    )

尝试 SPN: hive/hadoop.haddop_252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com@252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
>>> 失败: GSSError: SSPI: InitializeSecurityContext: ��ȫ����û�п��õ�ƾ֤


尝试 SPN: hive/hadoop.252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com@252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM
>>> 失败: GSSError: SSPI: InitializeSecurityContext: ��ȫ����û�п��õ�ƾ֤




RuntimeError: 所有 SPN 候选均失败，排查顺序：
 1) 票据: klist 确认有 krbtgt（密码模式确认密码正确）
 2) SPN : MRS Manager -> Hive -> 配置 -> 搜 principal，把准确值填进 SPN_HOST_CANDIDATES
 3) 网络: 本机到 HIVE_HOST:21066 与 KDC:88 是否可达
 4) 端口: MRS 上 HiveServer2 thrift 端口是否为 21066

In [5]:
# ================== 5. 验证查询 ==================
cursor = conn.cursor()
cursor.execute("SHOW DATABASES")
for row in cursor.fetchall():
    print(row)

AttributeError: 'NoneType' object has no attribute 'cursor'

In [6]:
# ================== 6. 业务示例（breast_cancer 表） ==================
cursor = conn.cursor()
cursor.execute("SHOW TABLES")
print("tables:", cursor.fetchall())

# cursor.execute("SELECT COUNT(*) FROM breast_cancer")
# print(cursor.fetchall())

conn.close()
print("done")

AttributeError: 'NoneType' object has no attribute 'cursor'